-------------------------------------
# **YLIVERTAINEN Bad Data Craniotomy**
-------------------------------------

----------------------
## ⚙️ **SETUP for Greatness** ⚙️
----------------------

In [1]:
from pathlib import Path
import sys
def find_root(marker: str = "predictive_modeling.py") -> Path:
    p = Path.cwd().resolve()
    for folder in [p, *p.parents]:
        if (folder / marker).is_file():
            return folder
    raise FileNotFoundError(f"Could not find {marker} above {p}")
root = find_root() # Path to .../ylivertainen
print("Current location:", root.name)
print(root)
repo_root = root.parent  # .../TheLibraryOfCode
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))
print("Inserted (times):", sys.path.count(str(root)))
print("Current location:", repo_root.name)

Current location: ylivertainen
C:\dev\TheLibraryOfCode\YlivertainenBadDataCraniotomy\ylivertainen
Inserted (times): 0
Current location: YlivertainenBadDataCraniotomy


In [2]:
%load_ext autoreload
%autoreload 2

In [3]:
#======================= IMPORT LIBRARIES ========================
#from ylivertainen.aesthetics_helpers import GREEN, YELLOW, ORANGE, RED, BOLD, BLUE, GRAY, RESET
import pandas as pd
from IPython.display import display, Markdown

#=======================🧹 DATA CLEANING 🧹========================
from ylivertainen.cleaning import pre_merge_check
from ylivertainen.cleaning import YlivertainenDataCleaningSurg

#=======================🚧 COHORT 🚧========================
from ylivertainen.config import big_beautiful_print
from ylivertainen.cohort import build_stroke_agreement_cohort

#=======================🙊 DDA 🙊========================
from ylivertainen.dda import YlivertainenDDA

#=======================🎨 EDA 🎨========================
from ylivertainen.eda import YlivertainenEDA

#=======================🔮 PREDICTIVE MODELING 🔮========================
from ylivertainen.predictive_modeling import build_model_frame

#=======================🧙‍♂️ INFERENTIAL 🧙‍♂️========================
from ylivertainen.INFERENTIAL import YlivertainenInferential

#=======================🤖 PREDICTIVE 🤖========================
from ylivertainen.PREDICTIVE import YlivertainenPredictive

#=======================💪🧠 AWESOME REPORT 💪🧠========================
from ylivertainen.THE_REPORT import YlivertainenTheReport

----------------------
# 🧹 **YLIVERTAINEN DATASURG** 🧹
----------------------

## **SETUP**

### Input CSVs and do a pre-merge check of all the columns

In [4]:
csvs = pre_merge_check(
    root=root,
    colname_length=25,
    show_dfs=False
    )

### Write the **[COLUMN_RENAME_MAP](../columns_to_canonical.py)**
- unified column renaming
- drops columns that are not specified
- DOES NOT create columns
### Pass CSVs into the **DataCleaningSurg**

In [5]:
cleaning = YlivertainenDataCleaningSurg(csvs)
print(cleaning)
display(cleaning.df.head())

ValueError: ❌ No CSV file/-s provided to merge_dfs

----------------------
## **WRITE AND APPLY [SCHEMA](../schema.py) and [DERIVED](../schema.py)**
----------------------

- dtypes
- categories (ordered and non-ordered)
- replace values
- make NaN
- keep or drop (decided after DERIVED acts)

**always match column names**

- match, timedelta, datetime bins
- base on *derive_from*
- specify *datetime_units* for datetime bins

**creates new cols**

**afterwards *keep=False* will delete specified cols**

In [ ]:
cleaned_df = cleaning.ylivertainen_janitor(
    apply_all=True,
    id_cols=['patient_card_no'],
    include_first=True,
    drop_dupes=False
    )

----------------------
## **VALUE OVERVIEW**
----------------------

In [ ]:
cleaning.explore_values()

---------------
---------------
---------------

--------------------------------------
# **🪪 INPUT TASKS 🪪**
--------------------------------------

In [ ]:
# import the tasks from ylivertainen.config
from ylivertainen.config import (TIA_MATCH, ISCHEMIC_STROKE_MATCH, ANY_CEREBROVASCULAR_MATCH)
# define all tasks here
all_tasks = TIA_MATCH, ISCHEMIC_STROKE_MATCH, ANY_CEREBROVASCULAR_MATCH
# define the current task
current_task = ISCHEMIC_STROKE_MATCH

--------------------------------------
# 🚧 **YLIVERTAINEN COHORT** 🚧
--------------------------------------

--------------------------
## **MODELLING TASKS**
--------------------------

print all the beautiful tasks to see their criteria and columns

In [ ]:
big_beautiful_print(current_task)
cohorted_df, metadata = build_stroke_agreement_cohort(cleaned_df, current_task, all_tasks)

---------------------
---------------------
---------------------

---------------
# 🙈 **YLIVERTAINEN DDA** 🙈
---------------

----------------------
## **LOAD DATASET**
----------------------

In [ ]:
descriptive = YlivertainenDDA(root, cohorted_df, metadata)
descriptive.DDA.columns

--------------------------------
### Exclude unneeded cols
--------------------------------

In [ ]:
if 'Matching TIA' in metadata['task_name']:
    specific_cols = ['nmpd_diag']
else:
    specific_cols = []

descriptive.keep_from_analysis(
    cols = ['patient_card_no', ] + specific_cols
    )

--------------------------------
## **DATASET OVERVIEW**
--------------------------------

In [ ]:
descriptive.dataset_overview(
    id_columns=['row_id']      # <== enter str for ID column name
    )

--------------------------------
## **UNIVARIATE SUMMARY**
--------------------------------

----------------------
### numerical
----------------------

In [ ]:
descriptive.analyse_numerical()

----------------------
### categorical
----------------------

In [ ]:
descriptive.analyse_categorical()

----------------------
### binary
----------------------

In [ ]:
descriptive.analyse_binary()

----------------------
### Export all the tables
----------------------

In [ ]:
descriptive.export_all_overviews(
    export=False
    )

-------------
# 📊🎨 **YLIVERTAINEN EDA** 🎨📊
-------------

----------------------
## **STUDY AIM AND QUESTIONS**
----------------------

-------------------------------------
### **COHORT DEFINITION**
- **Source:** PSKUS
- **Time window:** Year 2022-2023
- **Row:** 1 unique patient case
-------------------------------------
### **KEY INCLUSION SCOPE**
- "NMP pamata diagnoze": I63.9, I64, G45.9
- "Izrakstīšanas diagnozes kods"
-------------------------------------
### **STUDY AIM**
- **EXPLORE:** 
    - "NMP pamata diagnoze" vs "Izrakstīšanas diagnozes kods"
- **FIND:** 
    - Agreement/Mismatch
- **ADDITIONAL AIMS:**
    - Identify predictors associated with diagnostic agreement or mismatch
-------------------------------------
### **STUDY CONTEXT**

<p>This notebook explores agreement between prehospital stroke-related diagnosis and discharge diagnosis in PSKUS 2022-2023 data in patients.
<p>The goal is to understand how often prehospital diagnosis aligns with hospital discharge diagnosis and which patient or process factors may be associated with disagreement

-------------------------------------

----------------------
## **LOAD DATASET**
----------------------

In [ ]:
exploratory = YlivertainenEDA(root, cohorted_df, metadata)
display(exploratory.EDA.head(1))

----------------------
## **WHITELIST COLUMNS FOR ANALYSIS**
----------------------

In [ ]:
# ============ Defining features. Choosing columns to remain ============
exploratory.whitelist_columns(
    predictors = [
        'nmpd_diag', 'vecums', 'dzimums',
        'GKS', 'FastTest',
        'lidzPSKUS_timedelta_minutes',
        'izsaukuma_laiks_hour', 'izsaukuma_laiks_dow',
        'izsaukuma_laiks_workday_bool', 'izsaukuma_laiks_month_name',
        'GKS_missing', 'FastTest_missing'
        ],
    numerical_continuous = ['vecums']
    )

----------------------
## **ASSOCIATIONS & FEATURE DECISIONS**
----------------------

In [ ]:
if 'Matching TIA' in exploratory.task_name:
    leakage_vars = {'nmpd_diag',}
else:
    leakage_vars = None         # list of str
missingness_dict = None         # pd.DataFrame([{"column_name": "nmpd_diag", "missingness_type": "missing", "inferred_type": "categorical"}, {...}])
redundancy_pairs = None         # pd.DataFrame([{"var_a": "age", "var_b": "age_years", "preferred_keep": "age"}, {...}])

In [ ]:
exploratory.build_associations_table(leakage_vars)

In [ ]:
feature_decisions_df = exploratory.build_feature_decisions_table(missingness_dict, leakage_vars, redundancy_pairs)

----------------------
### EXPORT TABLES
----------------------

In [ ]:
exploratory.export_both_tables(
    export=False
    )

-------------
# 🔮 **YLIVERTAINEN PREDICTIVE MODELING** 🔮
-------------

In [ ]:
df_model = build_model_frame(
    root,
    cohorted_df,
    feature_decisions_df,
    metadata,
    export=False
    )

-------------
# 🧙‍♂️ **YLIVERTAINEN INFERENTIAL** 🧙‍♂️
-------------

In [ ]:
inferential = YlivertainenInferential(root, df_model, feature_decisions_df)
print(inferential)

In [ ]:
inferential.run_inferential(
    positive_class=0,
    random_state=42,
    clinical_relevance_floor="moderate",
    export=False
    )

-------------
# 🤖 **YLIVERTAINEN PREDICTIVE** 🤖
-------------

In [ ]:
predictive = YlivertainenPredictive(root, df_model, feature_decisions_df, default_positive_class=1)
print(predictive)

predictive_outputs = predictive.run_predictive(
    test_size=0.20,
    cv_folds=5,
    n_boot=1000,
    ci_level=0.95,
    show_tables=True,
    export=False,
    threshold_strategy="min_sensitivity",
    min_sensitivity_target=0.80
)

predictive_outputs["final_metrics"]

In [ ]:
predictive.predictive_summary()

-------------
# 💪🧠 **YLIVERTAINEN AWESOME REPORT** 💪🧠
-------------

In [ ]:

report = YlivertainenTheReport(
    root=root,
    metadata=metadata,
    cohorted_df=cohorted_df,
    feature_decisions_df=feature_decisions_df,
    df_model=df_model,
    predictive_outputs=predictive_outputs,
    # optional extras:
    dda_tables={
        "numerical": descriptive.numerical_DDA,
        "categorical": descriptive.categorical_DDA,
        "binary": descriptive.binary_DDA
        },
    associations_table=exploratory.associations_table,
    feature_importance_df=predictive.feature_importance_df,
)
report.generate()